In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import joblib
import tkinter as tk
from tkinter import messagebox

# --- Veri Yükleme ve Hazırlama ---
df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")
df["risk"] = df["Sleep Disorder"].apply(lambda x: 0 if x == "None" else 1)

if df["risk"].nunique() < 2 or df["risk"].sum() == len(df):
    healthy_data = {
        "Age": np.random.randint(25, 40, 100),
        "Sleep Duration": np.random.uniform(7.0, 8.5, 100),
        "Quality of Sleep": np.random.randint(7, 10, 100),
        "Physical Activity Level": np.random.randint(40, 70, 100),
        "Stress Level": np.random.randint(1, 3, 100),
        "Heart Rate": np.random.randint(60, 75, 100),
        "Daily Steps": np.random.randint(8000, 12000, 100),
        "Sleep Disorder": ["None"] * 100,
        "risk": [0] * 100
    }
    healthy_df = pd.DataFrame(healthy_data)
    df = pd.concat([df, healthy_df], ignore_index=True)

features = ["Age", "Sleep Duration", "Quality of Sleep",
            "Physical Activity Level", "Stress Level", "Heart Rate", "Daily Steps"]
X = df[features]
y = df["risk"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# Model başarı raporu
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC AUC Skoru:", roc_auc_score(y_test, y_prob))

# Model ve scaler'ı kaydet
joblib.dump(model, "model_xgb.pkl")
joblib.dump(scaler, "scaler.pkl")

# --- GUI Arayüzü ---
model = joblib.load("model_xgb.pkl")
scaler = joblib.load("scaler.pkl")
feature_order = features

root = tk.Tk()
root.title("Kalp Krizi Riski Tahmin Sistemi (XGBoost)")
entries = {}

for idx, feature in enumerate(feature_order):
    label = tk.Label(root, text=feature)
    label.grid(row=idx, column=0, padx=10, pady=5, sticky="e")
    entry = tk.Entry(root)
    entry.grid(row=idx, column=1, padx=10, pady=5)
    entries[feature] = entry

def predict_risk():
    try:
        user_input = np.array([[float(entries[feat].get()) for feat in feature_order]])
        user_scaled = scaler.transform(user_input)
        prediction = model.predict(user_scaled)[0]
        proba = model.predict_proba(user_scaled)[0][1]

        if prediction == 1:
            msg = f"⚠️ Yüksek risk tespit edildi!\nRisk olasılığı: %{proba*100:.2f}"
        else:
            msg = f"✅ Düşük risk: Güvendesiniz.\nRisk olasılığı: %{proba*100:.2f}"

        messagebox.showinfo("Tahmin Sonucu", msg)
    except Exception as e:
        messagebox.showerror("Hata", f"Tahmin yapılamadı:\n{str(e)}")

predict_button = tk.Button(root, text="Tahmini Hesapla", command=predict_risk, bg="lightblue")
predict_button.grid(row=len(feature_order), column=0, columnspan=2, pady=15)

root.mainloop()


/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [16:34:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        75

    accuracy                           1.00        95
   macro avg       1.00      1.00      1.00        95
weighted avg       1.00      1.00      1.00        95

ROC AUC Skoru: 1.0


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [4]:
!pip install xgboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 978.5 kB/s eta 0:00:00a 0:00:01
